<a href="https://colab.research.google.com/github/JulianNguyen05/DA_Worklify/blob/main/backend-ml/notebooks/train_cv_jd_scorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Worklify — Fine-tune model chấm điểm CV–JD

Notebook này fine-tune model embedding đa ngôn ngữ **có sẵn** (`paraphrase-multilingual-MiniLM-L12-v2`)
trên dữ liệu cặp (CV, JD, mức độ phù hợp), rồi xuất model để dùng trong `backend-ml`.

**Các bước**
1. Cài thư viện, nạp dữ liệu
2. Đo **baseline**: model có sẵn chấm điểm tốt đến đâu
3. **Fine-tune** trên tập train
4. Đo lại trên cùng tập test, so sánh trước/sau
5. Tính ngưỡng `SEMANTIC_SIM_LOW` / `SEMANTIC_SIM_HIGH` cho backend-ml
6. Lưu model, tải về và đặt vào `backend-ml/models/cv-jd-scorer/`

> Trên Colab: **Runtime → Change runtime type → T4 GPU** trước khi chạy.

In [1]:
# Ghim phiên bản để các thư viện tương thích nhau (Colab cài sẵn transformers bản mới hơn)
!pip install -q sentence-transformers==3.1.1 transformers==4.44.2 datasets==2.21.0 accelerate==0.34.2

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers


In [2]:
import os
import random
import shutil

import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr, spearmanr

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Colab:", IN_COLAB, "| Thiết bị:", DEVICE)

Colab: True | Thiết bị: cpu


## 1. Cấu hình

`DATA_SOURCE` chọn nguồn dữ liệu:

| Giá trị | Ý nghĩa |
|---|---|
| `"demo"` | Tự sinh dữ liệu giả để chạy thử toàn bộ notebook. **Kết quả không có giá trị báo cáo.** |
| `"upload"` | Tải file CSV từ máy lên Colab |
| `"drive"` | Đọc CSV từ Google Drive (`DRIVE_CSV_PATH`) |

File CSV (hoặc `.csv.gz`) cần 3 cột: `cv_text`, `jd_text`, `label` (số thực 0–1, càng cao càng phù hợp).

**Dataset gộp sẵn:** `backend-ml/data/cv_jd_pairs.csv.gz` (~11 nghìn cặp từ 4 bộ trên Hugging Face),
tạo bằng `backend-ml/scripts/build_dataset.py`. Chọn `"upload"` rồi tải file này lên, hoặc đặt
lên Google Drive và chọn `"drive"`.

In [3]:
DATA_SOURCE = "upload"            # "demo" | "upload" | "drive"
DRIVE_CSV_PATH = "/content/drive/MyDrive/worklify/cv_jd_pairs.csv.gz"

BASE_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
OUTPUT_DIR = "cv-jd-scorer"     # tên thư mục này phải trùng với backend-ml/models/cv-jd-scorer

EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
MAX_SEQ_LENGTH = 256            # số token tối đa mỗi văn bản khi train

## 2. Dữ liệu

Gợi ý nguồn dữ liệu thật:
- Bộ dữ liệu resume–job description công khai trên Kaggle / Hugging Face
- CV và JD mẫu tự soạn (có thể nhờ LLM sinh rồi tự gán nhãn lại)
- Dữ liệu từ chính Worklify: đơn ứng tuyển được nhà tuyển dụng `ACCEPTED` (nhãn cao) / `REJECTED` (nhãn thấp)

Ô dưới định nghĩa bộ sinh dữ liệu **demo** (chỉ dùng khi `DATA_SOURCE = "demo"`).

In [4]:
ROLES = {
    "backend":  {"title": "Backend Developer", "skills": ["Java", "Spring Boot", "MySQL", "Docker", "REST API", "Redis"]},
    "frontend": {"title": "Frontend Developer", "skills": ["ReactJS", "JavaScript", "TypeScript", "TailwindCSS", "Redux", "HTML/CSS"]},
    "mobile":   {"title": "Mobile Developer", "skills": ["Flutter", "Dart", "Kotlin", "Swift", "Firebase", "REST API"]},
    "data":     {"title": "Data Analyst", "skills": ["SQL", "Python", "Pandas", "Power BI", "Excel", "Thống kê"]},
    "devops":   {"title": "DevOps Engineer", "skills": ["Docker", "Kubernetes", "AWS", "Linux", "CI/CD", "Terraform"]},
    "account":  {"title": "Kế toán", "skills": ["Kế toán tổng hợp", "Báo cáo thuế", "Excel", "MISA", "Hoá đơn", "Công nợ"]},
    "sales":    {"title": "Nhân viên kinh doanh", "skills": ["Tư vấn khách hàng", "Đàm phán", "CRM", "Chăm sóc khách hàng", "Telesales", "Báo giá"]},
}
RELATED = {("backend", "devops"), ("backend", "frontend"), ("frontend", "mobile"), ("data", "backend"), ("account", "data")}


def make_cv(role_key, rng):
    r = ROLES[role_key]
    skills = rng.sample(r["skills"], k=rng.randint(3, 5))
    years = rng.randint(0, 6)
    return (f"{r['title']} với {years} năm kinh nghiệm. Kỹ năng: {', '.join(skills)}. "
            f"Đã tham gia các dự án sử dụng {skills[0]} và {skills[1]}.")


def make_jd(role_key, rng):
    r = ROLES[role_key]
    skills = rng.sample(r["skills"], k=4)
    return (f"Tuyển {r['title']}. Yêu cầu: thành thạo {', '.join(skills[:3])}; "
            f"biết {skills[3]} là lợi thế. Tối thiểu {rng.randint(1, 3)} năm kinh nghiệm.")


def make_demo_dataset(n=800, seed=SEED):
    rng = random.Random(seed)
    keys = list(ROLES)
    rows = []
    for _ in range(n):
        jd_role = rng.choice(keys)
        roll = rng.random()
        if roll < 0.4:
            cv_role, base = jd_role, 0.9
        elif roll < 0.6:
            related = [b for a, b in RELATED if a == jd_role] + [a for a, b in RELATED if b == jd_role]
            cv_role, base = (rng.choice(related), 0.5) if related else (jd_role, 0.9)
        else:
            cv_role = rng.choice([k for k in keys if k != jd_role])
            base = 0.5 if (cv_role, jd_role) in RELATED or (jd_role, cv_role) in RELATED else 0.1
        label = min(1.0, max(0.0, base + rng.uniform(-0.08, 0.08)))
        rows.append({"cv_text": make_cv(cv_role, rng), "jd_text": make_jd(jd_role, rng), "label": round(label, 3)})
    return pd.DataFrame(rows)

In [5]:
if DATA_SOURCE == "demo":
    df = make_demo_dataset()
elif DATA_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(next(iter(uploaded)))
elif DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    df = pd.read_csv(DRIVE_CSV_PATH)
else:
    raise ValueError(f"DATA_SOURCE không hợp lệ: {DATA_SOURCE}")

missing_cols = {"cv_text", "jd_text", "label"} - set(df.columns)
assert not missing_cols, f"CSV thiếu cột: {missing_cols}"

df = df.dropna(subset=["cv_text", "jd_text", "label"]).copy()
df["label"] = df["label"].astype(float).clip(0, 1)
print(f"Số mẫu: {len(df)}")
if "source" in df.columns:
    print(df["source"].value_counts())
print(df["label"].describe())
df.head()

Saving cv_jd_pairs.csv.gz to cv_jd_pairs.csv.gz
Số mẫu: 11367
source
cnamuangtoun    7743
scmlewis        2934
netsol           651
0xnbk             39
Name: count, dtype: int64
count    11367.000000
mean         0.428625
std          0.348696
min          0.000000
25%          0.100000
50%          0.494300
75%          0.900000
max          1.000000
Name: label, dtype: float64


,cv_text,jd_text,label,source,original_label
0,SummaryHighly motivated Sales Associate with e...,Net2Source Inc. is an award-winning total work...,0.1,cnamuangtoun,No Fit
1,Professional SummaryCurrently working with Cat...,At Salas OBrien we tell our clients that were ...,0.1,cnamuangtoun,No Fit
2,SummaryI started my construction career in Jun...,Schweitzer Engineering Laboratories (SEL) Infr...,0.1,cnamuangtoun,No Fit
3,SummaryCertified Electrical Foremanwith thirte...,"Mizick Miller & Company, Inc. is looking for a...",0.1,cnamuangtoun,No Fit
4,SummaryWith extensive experience in business/r...,Life at Capgemini\nCapgemini supports all aspe...,0.1,cnamuangtoun,No Fit


In [6]:
# Chia train / val / test = 80 / 10 / 10
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
n = len(df)
train_df = df.iloc[: int(n * 0.8)].reset_index(drop=True)
val_df = df.iloc[int(n * 0.8): int(n * 0.9)].reset_index(drop=True)
test_df = df.iloc[int(n * 0.9):].reset_index(drop=True)
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

train=9093  val=1137  test=1137


## 3. Baseline: model có sẵn

Hàm `score_pairs` tính cosine similarity **giống hệt cách backend-ml làm** (tách văn bản thành
đoạn ≤ 400 ký tự, embed từng đoạn rồi lấy trung bình), để số đo ở đây phản ánh đúng điểm trong hệ thống.

Chỉ số chính: **Spearman** giữa cosine và nhãn (càng gần 1 càng tốt: model xếp hạng đúng
cặp nào phù hợp hơn).

In [7]:
from sentence_transformers import SentenceTransformer


def chunk_text(text, max_chars=400):
    chunks, current = [], ""
    for line in str(text).splitlines():
        line = line.strip()
        if not line:
            continue
        if len(current) + len(line) + 1 > max_chars and current:
            chunks.append(current)
            current = ""
        while len(line) > max_chars:
            chunks.append(line[:max_chars])
            line = line[max_chars:]
        current = f"{current}\n{line}" if current else line
    if current:
        chunks.append(current)
    return chunks or [""]


def embed(model, text):
    vectors = model.encode(chunk_text(text), normalize_embeddings=True)
    mean = vectors.mean(axis=0)
    return mean / (np.linalg.norm(mean) or 1.0)


def score_pairs(model, frame):
    return np.array([float(embed(model, cv) @ embed(model, jd)) for cv, jd in zip(frame["cv_text"], frame["jd_text"])])


def evaluate(model, frame, name):
    sims = score_pairs(model, frame)
    result = {
        "model": name,
        "spearman": spearmanr(sims, frame["label"]).correlation,
        "pearson": pearsonr(sims, frame["label"])[0],
        "cosine_tb_phu_hop": sims[frame["label"] >= 0.7].mean(),
        "cosine_tb_khong_phu_hop": sims[frame["label"] <= 0.3].mean(),
    }
    return result, sims


base_model = SentenceTransformer(BASE_MODEL, device=DEVICE)
baseline_result, baseline_sims = evaluate(base_model, test_df, "Có sẵn (baseline)")
pd.DataFrame([baseline_result]).round(4)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

KeyboardInterrupt: 

## 4. Fine-tune

Dùng `CosineSimilarityLoss`: model học sao cho cosine similarity của cặp (CV, JD) tiến gần tới nhãn.

In [ ]:
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

model = SentenceTransformer(BASE_MODEL, device=DEVICE)
model.max_seq_length = MAX_SEQ_LENGTH

train_ds = Dataset.from_pandas(train_df[["cv_text", "jd_text", "label"]])
val_ds = Dataset.from_pandas(val_df[["cv_text", "jd_text", "label"]])

val_evaluator = EmbeddingSimilarityEvaluator(
    val_df["cv_text"].tolist(), val_df["jd_text"].tolist(), val_df["label"].tolist(), name="val"
)

args = SentenceTransformerTrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    fp16=DEVICE == "cuda",
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    report_to="none",
    seed=SEED,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    loss=losses.CosineSimilarityLoss(model),
    evaluator=val_evaluator,
)
trainer.train()

In [ ]:
finetuned_result, finetuned_sims = evaluate(model, test_df, "Đã fine-tune")
comparison = pd.DataFrame([baseline_result, finetuned_result]).round(4)
comparison

## 5. Ngưỡng cho backend-ml

backend-ml kéo giãn cosine similarity từ `[SEMANTIC_SIM_LOW, SEMANTIC_SIM_HIGH]` về thang 0–100.
Ô dưới đề xuất 2 ngưỡng dựa trên tập test: LOW ≈ cosine trung vị của cặp không phù hợp,
HIGH ≈ cosine trung vị của cặp rất phù hợp.

In [ ]:
low = float(np.median(finetuned_sims[test_df["label"] <= 0.3]))
high = float(np.median(finetuned_sims[test_df["label"] >= 0.7]))
print("Đặt các biến môi trường sau cho backend-ml:")
print(f"SEMANTIC_SIM_LOW={low:.2f}")
print(f"SEMANTIC_SIM_HIGH={high:.2f}")

## 6. Lưu và xuất model

In [ ]:
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
model.save(OUTPUT_DIR, create_model_card=False)  # model card không cần cho backend-ml
archive = shutil.make_archive(OUTPUT_DIR, "zip", root_dir=".", base_dir=OUTPUT_DIR)
print("Đã lưu:", archive)

# Kiểm tra: nạp lại model vừa lưu và chấm thử một cặp
reloaded = SentenceTransformer(OUTPUT_DIR, device=DEVICE)
sample = test_df.iloc[0]
print("Nhãn:", sample["label"], "| Cosine:", round(float(embed(reloaded, sample["cv_text"]) @ embed(reloaded, sample["jd_text"])), 4))

if IN_COLAB:
    from google.colab import files
    files.download(archive)

## 7. Đưa model vào Worklify

1. Giải nén `cv-jd-scorer.zip` vào `backend-ml/models/` (được thư mục `backend-ml/models/cv-jd-scorer/`).
2. Đặt `SEMANTIC_SIM_LOW` / `SEMANTIC_SIM_HIGH` ở mục 5 vào phần `environment` của service
   `backend-ml` trong `docker-compose.yml`.
3. Khởi động lại: `docker compose up -d --build backend-ml` (trong `backend-core/`).
4. Kiểm tra `http://localhost:8000/health` trả về `"model": "local:cv-jd-scorer"`.

Ghi bảng so sánh ở mục 4 vào báo cáo (nhớ dùng dữ liệu thật, không phải `"demo"`).